# Lab 5 - Large Hamming Weight Swings

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://learn.ChipWhisperer.io/courses/sca101). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous part of the course, you saw that a microcontroller's power consumption changes based on what it's doing. In the case of a simple password check, this allowed us to see how many characters of the password we had correct, eventually resulting in the password being broken.*

*That attack was based on different code execution paths showing up differently in power traces. In the previous video, you saw how, theoretically, the Hamming weight of data manipulated by the target should be present in our power traces. In this lab, we'll put that to the test.*


**LEARNING OUTCOMES:**

* Modifying data sent to a target device
* Detecting leakage using average traces

## Prerequisites

Hold up! Before you continue, check you've done the following tutorials:

* ☑ [Lab 4 - Power Analysis to Bypass Password](../Lab%204%20-%20Power%20Analysis%20to%20Bypass%20Password/Lab.ipynb).

## Intro

Now that we've seen the power of Simple Power Analysis, the goal of this lab will be to move onto the basics of Correlation Power Analysis. The first step of that is understanding how manipulating data
causes the target to consume different amounts of power. In particular, the previous video posited that there should be a linear relationship between the Hamming weight of the data being manipulated and the power consumed.

In this lab, we're going to simplify that a bit and see if we can see a difference between a number with a large Hamming weight, 0xFF, and a small Hamming weight, 0x00. These values have Hamming weights of 8 and 0, respectively, which is the largest difference for an 8-bit value.

## Setup

As usual, let's connect to our Husky and set it up. You should be a pro at this by now:

We're finally moving onto AES firmware for this lab, which we'll be using for most of the rest of SCA101. The firmware project for testing AES is `simpleserial-aes`. Let's build this and program the firmware onto our SAM4S.

In [ ]:
%%bash
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=CWHUSKY CRYPTO_TARGET=TINYAES128C SS_VER=SS_VER_1_1 -j

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-CWHUSKY.hex")

With that done, we can move onto capturing traces. Doing this is very similar to interacting with the `simpleserial-base` project from a few labs ago. Upon boot, we want to set an AES key - this is just to make sure the target is properly initialized and isn't doing something like using an all zero key. From there, things are extactly as with `simpleserial-base`: we send a `p` command and capture a power trace. Of course, on the target side, things are very different. Rather than just sending us back what we sent it, the target is encrypting the data we send using AES-128 and the key we set earlier, then sending that back instead.

We're going to make one additional change - we're going to always set the first byte of the plaintext to either 0x00, or 0xFF. As mentioned earlier, these are the two most extreme values of Hamming weight for an 8-bit number.

**Create your capture loop below:**

## Grouping Traces

Alright, so we've got our traces with large Hamming weight swings. Currently, these traces are all mixed up. **Separate them into two groups: `one_list` and `zero_list`:**

We should have two different lists. Whether we sent 0xFF or 0x00 was random, so these lists likely won't be evenly dispersed. Next, we'll want to take an average of each group (make sure you take an average of each trace at each point! We don't want an average of the traces in time), which will help smooth out any outliers and also fix our issue of having a different number of traces for each group.

**Finally, subtract the two averages and plot the resulting data:**

You should see a very distinct trace near the beginning of the plot, meaning that the data being manipulated in the target device is visible in its power trace! Again, there's a lot of room to explore here:

* Try setting multiple bytes to 0x00 and 0xFF.
* Try using smaller hamming weight differences. Is the spike still distinct? What about if you capture more traces?
* We focused on the first byte here. Try putting the difference plots for multiple different bytes on the same plot.
* The target is running AES here. Can you get the spikes to appear in different places if you set a byte in a later round of AES (say round 5) to 0x00 or 0xFF?

One other note that might trip you up: rememeber you are measuring the voltage at the input to ChipWhisperer, which is measured across a shunt resistor. The end result of this is that a lower voltage actually means more power. So you might see the spikes flipped from the "expected" direction based on a 1 taking more power than a 0.

**For fun, we can also plot our two average plots. These plots look very similar to our normal power trace, but there should be a spot where there's a large difference between the two. Can you find it?**

## Conclusions & Next Steps

You should be convinced that there's measurable data leakage in power traces and that this information has some relationship with the Hamming weight of the data manipulated.

In the next lab, we'll be investigation a more substantial and detailed model for this leakage.

In [ ]:
scope.dis()
target.dis()

---
<small>NO-FUN DISCLAIMER: This material is Copyright (C) NewAE Technology Inc., 2015-2026. ChipWhisperer is a trademark of NewAE Technology Inc., claimed in all jurisdictions, and registered in at least the United States of America, European Union, and Peoples Republic of China.

Tutorials derived from our open-source work must be released under the associated open-source license, and notice of the source must be *clearly displayed*. Only original copyright holders may license or authorize other distribution - while NewAE Technology Inc. holds the copyright for many tutorials, the github repository includes community contributions which we cannot license under special terms and **must** be maintained as an open-source release. Please contact us for special permissions (where possible).

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>